In [8]:
%%writefile requirements.txt
fastapi
uvicorn
pydantic
scikit-learn
pandas
numpy
joblib

Overwriting requirements.txt


In [9]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib
import uvicorn

# 1. Initialize App
app = FastAPI(title="Insurance Premium Predictor")

# 2. Load Model 
# (Ensure 'best_insurance_model.pkl' is in the same directory)
try:
    model = joblib.load("best_insurance_model.pkl")
except Exception as e:
    print(f"Error loading model: {e}")

# 3. Define Request Schema
class InsuranceData(BaseModel):
    age_group: str
    gender: str
    bmi_category: str
    smoker: int
    children: int
    income_lpa: float
    city: str
    occupation: str
    type_policy: str
    type_product: str
    tenure_years: float
    reimbursement: int

# 4. Endpoints
@app.get("/")
def health_check():
    return {"status": "Active", "message": "API is running."}

@app.post("/predict")
def predict_premium(data: InsuranceData):
    try:
        # Convert JSON payload directly into a DataFrame
        # model.predict expects a 2D array or DataFrame matching training features
        input_df = pd.DataFrame([data.dict()])
        
        prediction = model.predict(input_df)
        
        return {
            "status": "success",
            "predicted_premium": round(float(prediction[0]), 2)
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

Overwriting main.py


In [10]:
%%writefile Dockerfile
# Use lightweight Python base
FROM python:3.10-slim

# Set working directory inside container
WORKDIR /app

# Copy dependency file and install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model artifact
COPY main.py .
COPY best_insurance_model.pkl .

# Expose port for the FastAPI server
EXPOSE 8000

# Start the application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [11]:
!docker build -t insurance_api:latest .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 509B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.10-slim
#2 DONE 2.1s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 1.57kB 0.0s done
#4 DONE 0.0s

#5 [1/6] FROM docker.io/library/python:3.10-slim@sha256:fd76ade0c607f27677bc04be3c60749f400eedc941d9e72967e19a4cedff80c2
#5 resolve docker.io/library/python:3.10-slim@sha256:fd76ade0c607f27677bc04be3c60749f400eedc941d9e72967e19a4cedff80c2 0.0s done
#5 DONE 0.0s

#6 [3/6] COPY requirements.txt .
#6 CACHED

#7 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#7 CACHED

#8 [2/6] WORKDIR /app
#8 CACHED

#9 [5/6] COPY main.py .
#9 CACHED

#10 [6/6] COPY best_insurance_model.pkl .
#10 CACHED

#11 exporting to image
#11 exporting layers done
#11 exporting manifest sha256

In [12]:
!docker run -d -p 8000:8000 --name insurance_container insurance_api:latest

6e5bab66dd019a80a23523f243f9783ce45d074cbf4d967666279e98a4ccf654


In [13]:
import requests
import time

# Brief pause to ensure the container web server has fully started
time.sleep(3) 

url = "http://localhost:8000/predict"

# Sample row from your dataset
payload = {
    "age_group": "Adult",
    "gender": "M",
    "bmi_category": "Overweight",
    "smoker": 1,
    "children": 1,
    "income_lpa": 2.9,
    "city": "Amritsar",
    "occupation": "Electrician",
    "type_policy": "I",
    "type_product": "S",
    "tenure_years": 24.51,
    "reimbursement": 0
}

try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    print("API Request Successful!")
    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
except requests.exceptions.RequestException as e:
    print(f"API Request Failed: {e}")

API Request Successful!
Status Code: 200
Response: {'status': 'success', 'predicted_premium': 17007.15}


In [7]:
!docker stop insurance_container
!docker rm insurance_container

failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
